# jE_01 - Jarak antar-dataset, separabilitas, dan ketakcocokan normalisasi (JURNAL)

Analisis dari data yang **sudah ada** (tidak melatih model federated baru). Tiga konfigurasi:
`far@9` (CIC2018+UNSW, 9 fitur), `near@76` (CIC2017+CIC2018, 76 fitur), `near@9` (near dengan 9 fitur yang sama dgn far).

## Analisis A - jarak dan separabilitas
Naskah masih memakai label "near/far" tanpa ukuran jarak, dan menduga selisih sisa 0,077 (far vs near@9) berasal dari
**separabilitas** dataset. Di sini diukur:
1. **Jarak antar-client**: rata-rata jarak Wasserstein-1 per fitur (data di-winsorize 1-99% lalu distandarkan dengan statistik gabungan)
   dan **AUC pengklasifikasi domain** (seberapa mudah membedakan asal sampel; 0,5 = tak bisa, 1,0 = sangat mudah).
2. **Separabilitas internal** tiap client: AUC dan MCC pengklasifikasi pohon (serangan vs normal) dilatih di train client itu sendiri,
   diuji di test client itu sendiri.

**Kriteria dibaca (disepakati SEBELUM melihat angka):**
- Dugaan separabilitas **didukung** bila AUC internal UNSW (`far@9`) lebih tinggi >= 0,03 daripada AUC internal tertinggi di antara
  kedua client `near@9`. Bila selisih < 0,03, dugaan **tidak didukung**.
- Jarak (W1, AUC domain) dilaporkan secara **deskriptif**; tidak ada arah yang diharapkan.

## Analisis B - ketakcocokan normalisasi
Evaluasi "pooled" menormalisasi data tiap client dengan statistik gabungan, padahal model dilatih dengan statistik lokal. Untuk tiap client:
- `shift`  = rata-rata |mean_client - mean_gabungan| / std_gabungan  (pergeseran rata-rata)
- `scale`  = rata-rata |log(std_client / std_gabungan)|              (pergeseran skala)
- `mismatch` = shift + scale.
Lalu dibandingkan dengan **penurunan MCC per client** (MCC protokol lokal - MCC protokol pooled), dari hasil federated yang sudah ada (rata-rata seed).

**Kriteria dibaca (disepakati SEBELUM melihat angka):** 6 titik (3 konfigurasi x 2 client). Korelasi Spearman `mismatch` vs penurunan MCC:
- >= 0,89 (ambang signifikan 5% untuk n=6): indeks layak dipakai sebagai diagnostik.
- 0,60-0,89: indikasi lemah, hanya deskriptif.
- < 0,60: indeks tidak berguna.
Catatan: dua client dalam satu konfigurasi berbagi statistik gabungan, jadi keenam titik tidak sepenuhnya independen.

**Output:** `journal_out/analysis_*.csv`, `analysis_summary.json`, gambar PNG, S3 `federated_journal/analysis/`. Jalankan di SageMaker.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','boto3','scipy','pandas','scikit-learn','matplotlib'):
    mod='sklearn' if m=='scikit-learn' else m
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, json, csv, time, numpy as np, pandas as pd, boto3
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, matthews_corrcoef

BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
FAR_ROOT='unsw-far/federated_survey'
NEAR_ROOT='federated_journal/neardomain'
OUT='federated_journal/analysis'
OUTDIR='journal_out'; os.makedirs(OUTDIR,exist_ok=True)
FIGDIR=OUTDIR   # gambar disimpan di journal_out (Kiro menyalin ke figure-fed dari S3, hindari bentrok git pull)
s3=boto3.client('s3',region_name=REGION)
RS=0

NINE=['flow duration','total fwd packets','total backward packets',
      'total length of fwd packets','total length of bwd packets',
      'fwd packet length mean','bwd packet length mean','flow bytes/s','bwd packets/s']

def npz(key):
    return np.load(io.BytesIO(s3.get_object(Bucket=BUCKET,Key=key)['Body'].read()),allow_pickle=True)
def load_clients(root, clients):
    D={}; feats=None
    for c in clients:
        D[c]={}
        for sp in ('train','test'):
            d=npz(f'{root}/data/{c}/{c}_{sp}.npz')
            D[c][sp]=(np.asarray(d['X'],float),np.asarray(d['y'],int)); feats=[str(f) for f in d['feats']]
    return D,feats

FAR,far_feats=load_clients(FAR_ROOT,['cic','unsw'])
NEAR,near_feats=load_clients(NEAR_ROOT,['cic2017','cic2018'])
idx9=[near_feats.index(f) for f in NINE]      # akan error jelas bila nama fitur tak ada
NEAR9={c:{sp:(NEAR[c][sp][0][:,idx9],NEAR[c][sp][1]) for sp in ('train','test')} for c in NEAR}
CONFIGS={'far@9':(FAR,('cic','unsw')),'near@76':(NEAR,('cic2017','cic2018')),'near@9':(NEAR9,('cic2017','cic2018'))}
for k,(D,cl) in CONFIGS.items():
    print(k,{c:D[c]['train'][0].shape for c in cl})

In [ ]:
def sample(X,n,seed):
    rng=np.random.RandomState(seed)
    return X[rng.choice(len(X),size=min(n,len(X)),replace=False)]

def w1_mean(A,B,n=20000,seed=RS):
    """Wasserstein-1 rata-rata per fitur; winsorize 1-99% (gabungan) lalu standardisasi gabungan."""
    a=sample(A,n,seed); b=sample(B,n,seed+1); P=np.vstack([a,b])
    lo=np.percentile(P,1,axis=0); hi=np.percentile(P,99,axis=0)
    a=np.clip(a,lo,hi); b=np.clip(b,lo,hi); P=np.vstack([a,b]); mu=P.mean(0); sd=P.std(0); sd[sd==0]=1.0
    a=(a-mu)/sd; b=(b-mu)/sd
    w=np.array([stats.wasserstein_distance(a[:,j],b[:,j]) for j in range(a.shape[1])])
    return float(w.mean()), float(np.median(w))

def domain_auc(Atr,Btr,Ate,Bte,n=20000):
    a,b=sample(Atr,n,1),sample(Btr,n,2); X=np.vstack([a,b]); y=np.r_[np.zeros(len(a)),np.ones(len(b))]
    clf=HistGradientBoostingClassifier(max_iter=100,random_state=0).fit(X,y)
    a2,b2=sample(Ate,n,3),sample(Bte,n,4); X2=np.vstack([a2,b2]); y2=np.r_[np.zeros(len(a2)),np.ones(len(b2))]
    return float(roc_auc_score(y2,clf.predict_proba(X2)[:,1]))

def separability(Xtr,ytr,Xte,yte,ntr=60000,nte=100000):
    rng=np.random.RandomState(0)
    i=rng.choice(len(Xtr),size=min(ntr,len(Xtr)),replace=False); j=rng.choice(len(Xte),size=min(nte,len(Xte)),replace=False)
    clf=HistGradientBoostingClassifier(max_iter=100,random_state=0).fit(Xtr[i],ytr[i])
    p=clf.predict_proba(Xte[j])[:,1]
    return float(roc_auc_score(yte[j],p)), float(matthews_corrcoef(yte[j],(p>=0.5).astype(int)))

def mismatch(Xc,Xp):
    """Pergeseran statistik data client c bila dinormalisasi dgn statistik gabungan (Xp = train gabungan)."""
    mu_p=Xp.mean(0); sd_p=Xp.std(0); sd_p[sd_p==0]=1.0
    mu_c=Xc.mean(0); sd_c=Xc.std(0); sd_c[sd_c==0]=sd_p[sd_c==0]
    shift=float(np.mean(np.abs(mu_c-mu_p)/sd_p)); scale=float(np.mean(np.abs(np.log(sd_c/sd_p))))
    return shift,scale
print('fungsi analisis siap')

In [ ]:
# ===== ANALISIS A: jarak + separabilitas =====
rowsD=[]; rowsS=[]
for cfg,(D,cl) in CONFIGS.items():
    t0=time.time(); a,b=cl
    w_mean,w_med=w1_mean(D[a]['train'][0],D[b]['train'][0])
    dauc=domain_auc(D[a]['train'][0],D[b]['train'][0],D[a]['test'][0],D[b]['test'][0])
    rowsD.append(dict(config=cfg,clients=f'{a}|{b}',n_features=D[a]['train'][0].shape[1],
                      w1_mean=round(w_mean,4),w1_median=round(w_med,4),domain_auc=round(dauc,4)))
    for c in cl:
        auc,m=separability(*D[c]['train'],*D[c]['test'])
        rowsS.append(dict(config=cfg,client=c,n_features=D[c]['train'][0].shape[1],
                          attack_ratio_train=round(float(D[c]['train'][1].mean()),3),auc=round(auc,4),mcc=round(m,4)))
    print(f'{cfg} selesai [{time.time()-t0:.0f}s]')
dfD=pd.DataFrame(rowsD); dfS=pd.DataFrame(rowsS)
print('\n=== JARAK ANTAR-CLIENT ===\n',dfD.to_string(index=False))
print('\n=== SEPARABILITAS INTERNAL (pohon, serangan vs normal) ===\n',dfS.to_string(index=False))
unsw_auc=float(dfS[(dfS.config=='far@9')&(dfS.client=='unsw')].auc.iloc[0])
near9_best=float(dfS[dfS.config=='near@9'].auc.max())
print(f'\nKriteria separabilitas: AUC UNSW@9={unsw_auc:.3f} vs near@9 tertinggi={near9_best:.3f}; selisih={unsw_auc-near9_best:+.3f}')
print('  selisih >= 0.03 (mendukung) ?', (unsw_auc-near9_best)>=0.03)

In [ ]:
# ===== ANALISIS B: ketakcocokan normalisasi vs penurunan MCC per client =====
def read_json(key): return json.loads(s3.get_object(Bucket=BUCKET,Key=key)['Body'].read())
SEEDS=[13,42,101,202,303]
def near_drops(prefix):
    loc={}; poo={}
    for sd in SEEDS:
        r=read_json(f'{NEAR_ROOT}/{prefix}{sd}/eval_results.json')
        for c,m in r['federated_local']['per_client'].items(): loc.setdefault(c,[]).append(m['mcc'])
        for c,m in r['federated_pooled_diagnostic']['per_client'].items(): poo.setdefault(c,[]).append(m['mcc'])
    return {c:(float(np.mean(loc[c])),float(np.mean(poo[c]))) for c in loc}
def far_drops():
    d=pd.read_csv(io.BytesIO(s3.get_object(Bucket=BUCKET,Key=f'{FAR_ROOT}/results/r3_crossdataset_seeds.csv')['Body'].read()))
    f=d[d.setting=='federated-FedAvg']; out={}
    for c in ('cic','unsw'):
        out[c]=(float(f[(f.protocol=='local')&(f.scope==c)].mcc.mean()),
                float(f[(f.protocol=='pooled(diagnostic)')&(f.scope==c)].mcc.mean()))
    return out
DROPS={'far@9':far_drops(),'near@76':near_drops('run_J_NEAR_s'),'near@9':near_drops('run_J_NEAR9_s')}
rowsM=[]
for cfg,(D,cl) in CONFIGS.items():
    Xp=np.vstack([D[c]['train'][0] for c in cl])
    for c in cl:
        shift,scale=mismatch(D[c]['train'][0],Xp); loc,poo=DROPS[cfg][c]
        rowsM.append(dict(config=cfg,client=c,shift=round(shift,4),scale=round(scale,4),mismatch=round(shift+scale,4),
                          mcc_local=round(loc,4),mcc_pooled=round(poo,4),mcc_drop=round(loc-poo,4)))
dfM=pd.DataFrame(rowsM); print('=== KETAKCOCOKAN vs PENURUNAN MCC (per client) ===\n',dfM.to_string(index=False))
RHO={}
for col in ('shift','scale','mismatch'):
    r,p=stats.spearmanr(dfM[col],dfM.mcc_drop); RHO[col]=(float(r),float(p))
    print(f'Spearman {col:9s} vs mcc_drop: rho={r:+.3f} (p={p:.3f}, n={len(dfM)})')
rho=RHO['mismatch'][0]
print('\nKriteria (disepakati sebelum melihat angka):')
print('  rho >= 0.89 (layak sbg diagnostik) ?', rho>=0.89)
print('  0.60 <= rho < 0.89 (indikasi lemah) ?', 0.60<=rho<0.89)
print('  rho < 0.60 (tidak berguna) ?', rho<0.60)

In [ ]:
# ===== simpan + gambar + upload =====
dfD.to_csv(os.path.join(OUTDIR,'analysis_distance.csv'),index=False)
dfS.to_csv(os.path.join(OUTDIR,'analysis_separability.csv'),index=False)
dfM.to_csv(os.path.join(OUTDIR,'analysis_mismatch.csv'),index=False)
summary=dict(distance=rowsD,separability=rowsS,mismatch=rowsM,spearman={k:dict(rho=v[0],p=v[1]) for k,v in RHO.items()},
             sep_criterion=dict(unsw_auc_far9=unsw_auc,best_near9_auc=near9_best,diff=unsw_auc-near9_best))
json.dump(summary,open(os.path.join(OUTDIR,'analysis_summary.json'),'w'),indent=2)

plt.rcParams.update({'font.size':9,'figure.dpi':160,'savefig.bbox':'tight'})
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9.0,3.4))
lab=[f"{r.config}\n{r.client}" for r in dfS.itertuples()]
cols=['#2e8b57' if r.config=='far@9' else ('#3a6ea5' if r.config=='near@76' else '#d1495b') for r in dfS.itertuples()]
ax1.bar(range(len(dfS)),dfS.auc,color=cols); ax1.set_xticks(range(len(dfS))); ax1.set_xticklabels(lab,fontsize=6.5)
ax1.set_ylim(0.5,1.0); ax1.set_ylabel('within-client AUC (attack vs benign)'); ax1.set_title('(a) Internal separability',fontsize=9)
ax2.scatter(dfM.mismatch,dfM.mcc_drop,c=['#2e8b57' if c=='far@9' else ('#3a6ea5' if c=='near@76' else '#d1495b') for c in dfM.config],s=40)
for r in dfM.itertuples(): ax2.annotate(f"{r.config} {r.client}",(r.mismatch,r.mcc_drop),fontsize=6,xytext=(3,3),textcoords='offset points')
ax2.set_xlabel('normalization mismatch (shift + scale)'); ax2.set_ylabel('MCC drop (local - pooled eval)')
ax2.set_title(f"(b) Mismatch vs MCC drop (Spearman={RHO['mismatch'][0]:+.2f}, n={len(dfM)})",fontsize=9)
fig.tight_layout(); figpath=os.path.join(FIGDIR,'fig_analysis_mismatch.png'); fig.savefig(figpath); plt.close(fig)
print('gambar:',figpath)
for f in ('analysis_distance.csv','analysis_separability.csv','analysis_mismatch.csv','analysis_summary.json'):
    s3.upload_file(os.path.join(OUTDIR,f),BUCKET,f'{OUT}/results/{f}')
s3.upload_file(figpath,BUCKET,f'{OUT}/results/fig_analysis_mismatch.png')
print('upload ke S3 selesai')

In [ ]:
NEED=[f'{OUT}/results/{f}' for f in ('analysis_distance.csv','analysis_separability.csv','analysis_mismatch.csv',
      'analysis_summary.json','fig_analysis_mismatch.png')]
resp=s3.list_objects_v2(Bucket=BUCKET,Prefix=f'{OUT}/')
keys=sorted(o['Key'] for o in resp.get('Contents',[]))
print('File hasil di S3 (%d):'%len(keys))
for k in keys: print('  s3://%s/%s'%(BUCKET,k))
miss=[k for k in NEED if k not in keys]
print('LENGKAP' if not miss else 'KURANG: %s'%miss)
print()
print('Di komputer lokal (PowerShell), tarik dengan:')
print('  aws s3 sync s3://%s/%s/ ./analysis'%(BUCKET,OUT))